In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('injection_molding_predictive_maintenance.csv')
df.drop(columns=['timestamp'], inplace=True)
df.head(1)

,mold_id,mold_name,material_name,machine_tonnage,cycle_number,hopper_temp,nozzle_temp,H1,H2,H3,...,total_cycle_time,Hopper_State,Barrel_Heater_State,Screw_State,Injection_Unit_State,Hydraulic_System_State,Clamp_Unit_State,Mold_State,Cooling_System_State,Ejector_System_State
0,TL-2026-ABS-0892,Smart Router Enclosure (Top Cover) - 2 Cavity,"ABS (Flame Retardant, Medium Flow)",180,1,45.4,230,225,215,199,...,24.304,0,0,0,0,0,0,0,0,0


In [ ]:
target = ['Hopper_State','Barrel_Heater_State', 'Screw_State', 'Injection_Unit_State','Hydraulic_System_State', 
               'Clamp_Unit_State', 'Mold_State','Cooling_System_State', 'Ejector_System_State']  
sensor = [i for i in df.columns if i not in target]
sensor.remove('mold_name')
sensor.remove('mold_id')
sensor.remove('material_name')
len(sensor)

44

In [ ]:
def find_correlation(corr, threshold=0.95):
    """Iteratively drop the more globally-redundant column of the most-correlated
    remaining pair, until no pair exceeds threshold. Avoids over-pruning chains."""
    corr = corr.abs().copy()
    vals = corr.values.copy()
    np.fill_diagonal(vals, 0)  # ignore self-correlation
    corr = pd.DataFrame(vals, index=corr.index, columns=corr.columns)
    to_drop = []
    while True:
        max_corr = corr.values.max()
        if max_corr <= threshold or corr.shape[0] <= 1:
            break
        i, j = np.unravel_index(np.argmax(corr.values), corr.shape)
        col_i, col_j = corr.index[i], corr.columns[j]
        # drop whichever has the higher average correlation with everything else remaining
        drop_col = col_i if corr[col_i].mean() > corr[col_j].mean() else col_j
        to_drop.append(drop_col)
        corr = corr.drop(index=drop_col, columns=drop_col)
    return to_drop

# correlation-based pruning of redundant raw sensor columns
corr = df[sensor].corr()
to_drop = find_correlation(corr, threshold=0.95)
kept_numeric_cols = [c for c in sensor if c not in to_drop]
df.drop(columns=to_drop, inplace=True)
print(f"Dropped {len(to_drop)} redundant (|r|>0.95) columns, kept {len(kept_numeric_cols)}:")
print(kept_numeric_cols)

Dropped 25 redundant (|r|>0.95) columns, kept 16:
['cycle_number', 'hopper_temp', 'H1', 'mold_cavity_temp', 'inj_speed3', 'vp_transition', 'holding_pressure', 'back_pressure', 'decompression_speed', 'decompression_distance', 'cushion_position', 'mold_protection_pressure', 'ejection_stroke', 'part_weight', 'mould_close_time', 'clamp_close_position']


## Train Test Split

In [7]:
import pandas as pd
 
train_list = []
val_list = []
test_list = []
 
# Sort first
df = df.sort_values(['mold_id', 'cycle_number']).reset_index(drop=True)
 
for mold_id, group in df.groupby('mold_id'):
 
    n = len(group)
 
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
 
    train_list.append(group.iloc[:train_end])
    val_list.append(group.iloc[train_end:val_end])
    test_list.append(group.iloc[val_end:])
 
train_df = pd.concat(train_list).reset_index(drop=True)
val_df = pd.concat(val_list).reset_index(drop=True)
test_df = pd.concat(test_list).reset_index(drop=True)
 
print(f"Train: {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test: {test_df.shape}")


Train: (28000, 28)
Validation: (6000, 28)
Test: (6000, 28)


## Feature Engineering

In [8]:
window = 5
temp_pressure  = ['hopper_temp', 'H1', 'mold_cavity_temp', 'inj_speed3','holding_pressure', 'back_pressure', 'decompression_speed','mold_protection_pressure', 'ejection_stroke', 'part_weight', 'mould_close_time']

def add_engineered_features(data, window=window, temp_pressure=temp_pressure, kept_numeric_cols=kept_numeric_cols):
    data = data.sort_values(['mold_id', 'cycle_number']).reset_index(drop=True)
    grp = data.groupby('mold_id')

    # rolling mean & std
    roll_mean = grp[temp_pressure].rolling(window, min_periods=1).mean().droplevel(0)
    roll_mean.columns = [f'{c}_roll_mean' for c in temp_pressure]

    roll_std = grp[temp_pressure].rolling(window, min_periods=1).std().droplevel(0)
    roll_std.columns = [f'{c}_roll_std' for c in temp_pressure]

    # delta
    delta = grp[kept_numeric_cols].diff()
    delta.columns = [f'{c}_delta' for c in kept_numeric_cols]

    eng = pd.concat([roll_mean, roll_std, delta], axis=1).sort_index().fillna(0.0)
    return pd.concat([data, eng], axis=1)

train_df = add_engineered_features(train_df)
val_df = add_engineered_features(val_df)
test_df = add_engineered_features(test_df)

print(f"Train: {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test: {test_df.shape}")

Train: (28000, 66)
Validation: (6000, 66)
Test: (6000, 66)


In [13]:
#ohe on mold id
def add_mold_dummies(data):
    enc = pd.get_dummies(data['mold_id'], prefix='mold_id', dtype=int)
    return pd.concat([enc,data], axis=1)

train_df = add_mold_dummies(train_df)
val_df = add_mold_dummies(val_df)
test_df = add_mold_dummies(test_df)

In [ ]:
# scaling numeric features
scaler = StandardScaler()
id_cat_cols = ['mold_id', 'mold_name', 'material_name', 'cycle_number']
mold_dummy_cols = [c for c in train_df.columns if c.startswith('mold_id_')]

exclude_cols = id_cat_cols + target + mold_dummy_cols

numeric_cols = train_df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in exclude_cols]

train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
val_df[numeric_cols] = scaler.transform(val_df[numeric_cols])
test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

In [15]:
train_df.drop(columns=['mold_id','mold_name','material_name'],inplace=True)
val_df.drop(columns=['mold_id','mold_name','material_name'],inplace=True)
test_df.drop(columns=['mold_id','mold_name','material_name'],inplace=True)

In [20]:
def reorder_target_last(data, target=target):
    cols = [c for c in data.columns if c not in target] + target
    return data[cols]

train_df = reorder_target_last(train_df)
val_df = reorder_target_last(val_df)
test_df = reorder_target_last(test_df)

X_train, y_train = train_df.drop(columns=target), train_df[target]
X_val, y_val = val_df.drop(columns=target), val_df[target]
X_test, y_test = test_df.drop(columns=target), test_df[target]